# Complexity Analysis — Entropy & Hurst Exponent

We measure the **complexity** and **long-range dependence** of four time-series datasets using:

| Metric | What it measures |
|--------|------------------|
| **Approximate Entropy (ApEn)** | Regularity / predictability — lower = more regular |
| **Sample Entropy (SampEn)** | Bias-corrected version of ApEn — lower = more regular |
| **Hurst Exponent (H)** | Long-range dependence — H=0.5 random, H>0.5 persistent, H<0.5 anti-persistent |

### Datasets
- Dow Jones (monthly closings)
- S&P 500 (daily opens)
- Lake Erie (monthly water levels)
- Monthly Milk Production (lbs/cow)

In [ ]:
# ============================================================
# PROCESS IDENTIFICATION
# ============================================================
import os
print(f"Process ID (PID): {os.getpid()}")


In [ ]:
# ============================================================
# NOTEBOOK TIMER — START
# ============================================================
import time as _timer_module
_NOTEBOOK_START_TIME = _timer_module.time()
print(f"Notebook execution started at: {_timer_module.strftime('%Y-%m-%d %H:%M:%S')}")


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy import stats as sp_stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# ── Dark-theme style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  '#0f0f1a',
    'axes.facecolor':    '#1a1a2e',
    'axes.edgecolor':    '#3a3a5c',
    'axes.labelcolor':   '#e0e0ff',
    'axes.titlesize':    13,
    'axes.labelsize':    11,
    'xtick.color':       '#a0a0cc',
    'ytick.color':       '#a0a0cc',
    'grid.color':        '#2a2a4a',
    'grid.linestyle':    '--',
    'grid.alpha':        0.6,
    'text.color':        '#e0e0ff',
    'font.family':       'DejaVu Sans',
    'lines.linewidth':   1.6,
})

PALETTE = {
    'Dow Jones':               '#7c83fd',
    'S&P 500':                 '#fc5c7d',
    'Lake Erie':               '#43e97b',
    'Monthly Milk Production': '#f7971e',
}

print('Libraries loaded ✓')

In [ ]:
# ── Load datasets ─────────────────────────────────────────────────────────────
def load_dataset(path):
    df = pd.read_csv(path)
    return df.iloc[:, 0], df.iloc[:, 1].astype(float).values

datasets = {
    'Dow Jones': {
        'path': '../content/monthly-closings-of-the-dowjones.csv',
        'unit': 'Index Points',
    },
    'S&P 500': {
        'path': '../content/sp500.csv',
        'unit': 'Open Price (USD)',
    },
    'Lake Erie': {
        'path': '../content/monthly-lake-erie-levels-1921-19.csv',
        'unit': 'Water Level (ft)',
    },
    'Monthly Milk Production': {
        'path': '../content/monthly-milk-production-pounds-p.csv',
        'unit': 'Pounds per Cow',
    },
}

for name, meta in datasets.items():
    idx, vals = load_dataset(meta['path'])
    meta['index']  = idx
    meta['values'] = vals
    print(f'{name:30s} → {len(vals):4d} observations')

print('\nAll datasets loaded ✓')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# APPROXIMATE ENTROPY (ApEn)
# ══════════════════════════════════════════════════════════════════════════════

def approximate_entropy(U, m=2, r_factor=0.2):
    """
    Compute Approximate Entropy (ApEn) of a time series.

    Parameters
    ----------
    U : 1-D array — the time series
    m : int        — embedding dimension (pattern length)
    r_factor : float — tolerance as fraction of std(U)

    Returns
    -------
    ApEn value (float)
    """
    U = np.array(U, dtype=np.float64)
    N = len(U)
    r = r_factor * np.std(U)

    def _phi(m_):
        # Build template vectors of length m_
        patterns = np.array([U[i:i+m_] for i in range(N - m_ + 1)])
        n_pat = len(patterns)
        counts = np.zeros(n_pat)
        for i in range(n_pat):
            # Chebyshev distance
            dists = np.max(np.abs(patterns - patterns[i]), axis=1)
            counts[i] = np.sum(dists <= r) / n_pat
        return np.mean(np.log(counts + 1e-30))

    return _phi(m) - _phi(m + 1)

print('ApEn function defined ✓')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SAMPLE ENTROPY (SampEn)
# ══════════════════════════════════════════════════════════════════════════════

def sample_entropy(U, m=2, r_factor=0.2):
    """
    Compute Sample Entropy (SampEn) — bias-corrected version of ApEn.

    Parameters
    ----------
    U : 1-D array — the time series
    m : int        — embedding dimension
    r_factor : float — tolerance as fraction of std(U)

    Returns
    -------
    SampEn value (float)
    """
    U = np.array(U, dtype=np.float64)
    N = len(U)
    r = r_factor * np.std(U)

    def _count_matches(m_):
        patterns = np.array([U[i:i+m_] for i in range(N - m_)])
        n_pat = len(patterns)
        count = 0
        for i in range(n_pat):
            for j in range(i + 1, n_pat):
                if np.max(np.abs(patterns[i] - patterns[j])) <= r:
                    count += 1
        return count

    A = _count_matches(m + 1)  # matches of length m+1
    B = _count_matches(m)      # matches of length m

    if B == 0:
        return float('inf')  # undefined
    return -np.log(A / B) if A > 0 else float('inf')

print('SampEn function defined ✓')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# HURST EXPONENT — Rescaled Range (R/S) method
# ══════════════════════════════════════════════════════════════════════════════

def hurst_exponent(ts):
    """
    Estimate the Hurst Exponent using the Rescaled Range (R/S) method.

    Returns
    -------
    H : float — Hurst exponent
    c : float — intercept
    log_n : array — log of window sizes
    log_rs : array — log of R/S values
    r_squared : float — R² of the fit
    """
    ts = np.array(ts, dtype=np.float64)
    N = len(ts)

    # Generate window sizes (powers of 2 that fit)
    max_k = int(np.floor(np.log2(N)))
    window_sizes = [2**i for i in range(3, max_k + 1)]

    rs_values = []
    for n in window_sizes:
        n_windows = N // n
        if n_windows < 1:
            continue
        rs_list = []
        for w in range(n_windows):
            segment = ts[w * n : (w + 1) * n]
            mean_seg = np.mean(segment)
            deviations = np.cumsum(segment - mean_seg)
            R = np.max(deviations) - np.min(deviations)
            S = np.std(segment, ddof=1)
            if S > 0:
                rs_list.append(R / S)
        if rs_list:
            rs_values.append((n, np.mean(rs_list)))

    if len(rs_values) < 2:
        return None, None, None, None, None

    log_n  = np.log(np.array([v[0] for v in rs_values]))
    log_rs = np.log(np.array([v[1] for v in rs_values]))

    # Linear regression: log(R/S) = H * log(n) + c
    slope, intercept, r_value, p_value, std_err = sp_stats.linregress(log_n, log_rs)

    return slope, intercept, log_n, log_rs, r_value**2


def hurst_exponent_dfa(ts):
    """
    Detrended Fluctuation Analysis (DFA) for Hurst exponent.
    Provides a second estimate for comparison.
    """
    ts = np.array(ts, dtype=np.float64)
    N = len(ts)
    mean_ts = np.mean(ts)
    profile = np.cumsum(ts - mean_ts)

    max_k = int(np.floor(np.log2(N)))
    window_sizes = [2**i for i in range(2, max_k)]

    fluct = []
    for n in window_sizes:
        n_windows = N // n
        if n_windows < 1:
            continue
        rms_list = []
        for w in range(n_windows):
            segment = profile[w * n : (w + 1) * n]
            x = np.arange(n)
            coeffs = np.polyfit(x, segment, 1)
            trend = np.polyval(coeffs, x)
            rms = np.sqrt(np.mean((segment - trend) ** 2))
            rms_list.append(rms)
        if rms_list:
            fluct.append((n, np.mean(rms_list)))

    if len(fluct) < 2:
        return None, None

    log_n = np.log(np.array([f[0] for f in fluct]))
    log_f = np.log(np.array([f[1] for f in fluct]))

    slope, intercept, r_value, p_value, std_err = sp_stats.linregress(log_n, log_f)
    return slope, r_value**2

print('Hurst exponent functions (R/S + DFA) defined ✓')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ADDITIONAL COMPLEXITY METRICS
# ══════════════════════════════════════════════════════════════════════════════

def permutation_entropy(ts, order=3, delay=1):
    """
    Compute Permutation Entropy (PE) — measures complexity via ordinal patterns.
    Normalised to [0, 1]:  0 = perfectly regular, 1 = completely random.
    """
    ts = np.array(ts, dtype=np.float64)
    N = len(ts)
    n_patterns = N - (order - 1) * delay

    if n_patterns <= 0:
        return None

    from collections import Counter
    pattern_counts = Counter()
    for i in range(n_patterns):
        indices = [i + j * delay for j in range(order)]
        pattern = tuple(np.argsort([ts[idx] for idx in indices]))
        pattern_counts[pattern] += 1

    probs = np.array(list(pattern_counts.values())) / n_patterns
    pe = -np.sum(probs * np.log2(probs))
    max_entropy = np.log2(__import__("math").factorial(order))
    return pe / max_entropy  # normalised


def lempel_ziv_complexity(ts, threshold='median'):
    """
    Lempel-Ziv complexity — binary sequence complexity.
    Higher = more complex / random.
    """
    ts = np.array(ts, dtype=np.float64)
    if threshold == 'median':
        t = np.median(ts)
    else:
        t = np.mean(ts)

    binary = ''.join(['1' if x >= t else '0' for x in ts])
    N = len(binary)

    # Count distinct substrings (Kasai-like)
    i = 0
    c = 1
    k = 1
    k_max = 1
    while i + k <= N:
        if binary[i + k - 1] in binary[0:i + k_max]:
            k += 1
        else:
            c += 1
            i += k
            k = 1
            k_max = k
        if i + k > N:
            break
        k_max = max(k_max, k)

    # Normalise by random sequence expectation
    b = 2  # binary alphabet
    lz_norm = c / (N / np.log2(N)) if N > 0 else 0
    return c, lz_norm

print('Additional complexity functions defined ✓')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# COMPUTE ALL METRICS
# ══════════════════════════════════════════════════════════════════════════════

for name, meta in datasets.items():
    vals = meta['values']
    print(f'\nComputing complexity for {name} ({len(vals)} points)...')

    # ── Approximate Entropy ───────────────────────────────────────────────
    apen_m2_r02 = approximate_entropy(vals, m=2, r_factor=0.2)
    apen_m2_r015 = approximate_entropy(vals, m=2, r_factor=0.15)
    apen_m3_r02 = approximate_entropy(vals, m=3, r_factor=0.2)
    print(f'  ApEn done')

    # ── Sample Entropy ────────────────────────────────────────────────────
    sampen_m2_r02 = sample_entropy(vals, m=2, r_factor=0.2)
    sampen_m2_r015 = sample_entropy(vals, m=2, r_factor=0.15)
    print(f'  SampEn done')

    # ── Hurst Exponent (R/S) ──────────────────────────────────────────────
    h_rs, h_rs_intercept, h_rs_logn, h_rs_logrs, h_rs_r2 = hurst_exponent(vals)
    print(f'  Hurst (R/S) done')

    # ── Hurst Exponent (DFA) ──────────────────────────────────────────────
    h_dfa, h_dfa_r2 = hurst_exponent_dfa(vals)
    print(f'  Hurst (DFA) done')

    # ── Permutation Entropy ───────────────────────────────────────────────
    pe_3 = permutation_entropy(vals, order=3)
    pe_4 = permutation_entropy(vals, order=4)
    pe_5 = permutation_entropy(vals, order=5)
    print(f'  PermEn done')

    # ── Lempel-Ziv Complexity ─────────────────────────────────────────────
    lz_count, lz_norm = lempel_ziv_complexity(vals)
    print(f'  LZ complexity done')

    # Store results
    meta['complexity'] = {
        'apen_m2_r02':      apen_m2_r02,
        'apen_m2_r015':     apen_m2_r015,
        'apen_m3_r02':      apen_m3_r02,
        'sampen_m2_r02':    sampen_m2_r02,
        'sampen_m2_r015':   sampen_m2_r015,
        'hurst_rs':         h_rs,
        'hurst_rs_r2':      h_rs_r2,
        'hurst_rs_logn':    h_rs_logn,
        'hurst_rs_logrs':   h_rs_logrs,
        'hurst_dfa':        h_dfa,
        'hurst_dfa_r2':     h_dfa_r2,
        'perm_entropy_3':   pe_3,
        'perm_entropy_4':   pe_4,
        'perm_entropy_5':   pe_5,
        'lz_count':         lz_count,
        'lz_norm':          lz_norm,
    }
    print(f'  ✓ All metrics computed for {name}')

print('\n══ All complexity metrics computed ══')

In [ ]:
# ── Printed numerical results ─────────────────────────────────────────────────
SEP  = '═' * 88
SEP2 = '─' * 88

for name, meta in datasets.items():
    c = meta['complexity']
    N = len(meta['values'])

    print(f'\n{SEP}')
    print(f'  {name.upper()}  —  Complexity Analysis')
    print(f'  N = {N}')
    print(SEP)

    # ── Entropy metrics ───────────────────────────────────────────────────
    print(f'\n  ENTROPY METRICS:')
    print(f'  {SEP2}')
    print(f'  {"Metric":<40} {"Value":>12} {"Interpretation"}')
    print(f'  {SEP2}')

    # ApEn
    v = c['apen_m2_r02']
    interp = 'highly regular' if v < 0.1 else 'regular' if v < 0.5 else 'moderate' if v < 1.0 else 'complex/random'
    print(f'  {"ApEn (m=2, r=0.2σ)":<40} {v:>12.6f} → {interp}')

    v = c['apen_m2_r015']
    interp = 'highly regular' if v < 0.1 else 'regular' if v < 0.5 else 'moderate' if v < 1.0 else 'complex/random'
    print(f'  {"ApEn (m=2, r=0.15σ)":<40} {v:>12.6f} → {interp}')

    v = c['apen_m3_r02']
    interp = 'highly regular' if v < 0.1 else 'regular' if v < 0.5 else 'moderate' if v < 1.0 else 'complex/random'
    print(f'  {"ApEn (m=3, r=0.2σ)":<40} {v:>12.6f} → {interp}')

    # SampEn
    v = c['sampen_m2_r02']
    if v == float('inf'):
        print(f'  {"SampEn (m=2, r=0.2σ)":<40} {"inf":>12} → no matching patterns')
    else:
        interp = 'highly regular' if v < 0.1 else 'regular' if v < 0.5 else 'moderate' if v < 1.0 else 'complex/random'
        print(f'  {"SampEn (m=2, r=0.2σ)":<40} {v:>12.6f} → {interp}')

    v = c['sampen_m2_r015']
    if v == float('inf'):
        print(f'  {"SampEn (m=2, r=0.15σ)":<40} {"inf":>12} → no matching patterns')
    else:
        interp = 'highly regular' if v < 0.1 else 'regular' if v < 0.5 else 'moderate' if v < 1.0 else 'complex/random'
        print(f'  {"SampEn (m=2, r=0.15σ)":<40} {v:>12.6f} → {interp}')

    # Permutation Entropy
    for order, key in [(3, 'perm_entropy_3'), (4, 'perm_entropy_4'), (5, 'perm_entropy_5')]:
        v = c[key]
        if v is not None:
            interp = 'deterministic' if v < 0.3 else 'structured' if v < 0.7 else 'complex' if v < 0.9 else 'near-random'
            print(f'  {f"Permutation Entropy (order={order})":<40} {v:>12.6f} → {interp}')

    # LZ complexity
    print(f'  {"Lempel-Ziv Complexity (raw count)":<40} {c["lz_count"]:>12d}')
    print(f'  {"Lempel-Ziv Complexity (normalised)":<40} {c["lz_norm"]:>12.6f} → {"random-like" if c["lz_norm"] > 0.7 else "structured" if c["lz_norm"] > 0.3 else "very regular"}')

    # ── Hurst Exponent ────────────────────────────────────────────────────
    print(f'\n  HURST EXPONENT (Long-range Dependence):')
    print(f'  {SEP2}')

    h = c['hurst_rs']
    if h is not None:
        if h > 0.5:
            h_interp = f'PERSISTENT (trending) — past increases predict future increases'
        elif h < 0.5:
            h_interp = f'ANTI-PERSISTENT (mean-reverting) — tends to reverse direction'
        else:
            h_interp = f'RANDOM WALK — no long-range dependence'
        print(f'  Hurst (R/S method):      H = {h:.6f}   R² = {c["hurst_rs_r2"]:.6f}   → {h_interp}')
    else:
        print(f'  Hurst (R/S method):      Could not compute (insufficient data)')

    h_dfa = c['hurst_dfa']
    if h_dfa is not None:
        if h_dfa > 0.5:
            dfa_interp = 'PERSISTENT'
        elif h_dfa < 0.5:
            dfa_interp = 'ANTI-PERSISTENT'
        else:
            dfa_interp = 'RANDOM WALK'
        print(f'  Hurst (DFA method):      H = {h_dfa:.6f}   R² = {c["hurst_dfa_r2"]:.6f}   → {dfa_interp}')
    else:
        print(f'  Hurst (DFA method):      Could not compute')

    # ── Overall interpretation ────────────────────────────────────────────
    print(f'\n  OVERALL INTERPRETATION:')
    print(f'  {SEP2}')

    # Entropy verdict
    apen = c['apen_m2_r02']
    sampen = c['sampen_m2_r02'] if c['sampen_m2_r02'] != float('inf') else apen
    avg_entropy = (apen + sampen) / 2
    if avg_entropy < 0.1:
        print('  • REGULARITY:     Very high — highly predictable / periodic signal')
    elif avg_entropy < 0.5:
        print('  • REGULARITY:     High — significant structure / low complexity')
    elif avg_entropy < 1.0:
        print('  • REGULARITY:     Moderate — some structure but with complexity')
    else:
        print('  • REGULARITY:     Low — high complexity / approach random behaviour')

    # Hurst verdict
    if h is not None:
        if h > 0.8:
            print(f'  • LONG-MEMORY:    Very strong persistence (H={h:.4f}) — strong trending behaviour')
        elif h > 0.6:
            print(f'  • LONG-MEMORY:    Moderate persistence (H={h:.4f}) — some trending')
        elif h > 0.4:
            print(f'  • LONG-MEMORY:    Near random walk (H={h:.4f}) — weak dependence')
        else:
            print(f'  • LONG-MEMORY:    Anti-persistent (H={h:.4f}) — mean-reverting dynamics')

    # Combined verdict
    pe = c['perm_entropy_5'] or c['perm_entropy_4'] or c['perm_entropy_3']
    if pe is not None and h is not None:
        if pe < 0.5 and h > 0.7:
            print('  • NATURE:         Structured + persistent → deterministic trend with clear patterns')
        elif pe > 0.8 and h > 0.6:
            print('  • NATURE:         Complex but persistent → noisy signal with underlying trend')
        elif pe > 0.8 and abs(h - 0.5) < 0.1:
            print('  • NATURE:         Random-walk-like → efficient market / unpredictable')
        elif pe < 0.5 and abs(h - 0.5) < 0.1:
            print('  • NATURE:         Structured but no long memory → periodic / cyclic')
        else:
            print(f'  • NATURE:         Mixed characteristics (PE={pe:.3f}, H={h:.3f})')

print(f'\n{SEP}')
print('  REFERENCE VALUES')
print('  • ApEn/SampEn:  0 = perfectly regular | >1 = very complex/random')
print('  • Hurst:        0 = anti-persistent | 0.5 = random walk | 1 = perfectly persistent')
print('  • PermEn:       0 = deterministic | 1 = completely random')
print('  • LZ (norm):    0 = trivial | ~1 = random-like complexity')
print(f'{SEP}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT — Hurst R/S log-log fits
# ══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Hurst Exponent — R/S Log-Log Regression', fontsize=16,
             fontweight='bold', color='#e0e0ff')

for ax, (name, meta) in zip(axes.flatten(), datasets.items()):
    c     = meta['complexity']
    color = PALETTE[name]

    log_n  = c['hurst_rs_logn']
    log_rs = c['hurst_rs_logrs']
    H      = c['hurst_rs']
    r2     = c['hurst_rs_r2']

    if log_n is not None:
        ax.scatter(log_n, log_rs, color=color, s=60, zorder=5,
                   edgecolors='white', linewidth=0.5)
        # Fit line
        fit_x = np.linspace(log_n.min(), log_n.max(), 100)
        fit_y = H * fit_x + c['hurst_rs_r2']  # approximate
        # Better: use actual fit
        from scipy.stats import linregress
        sl, ic, _, _, _ = linregress(log_n, log_rs)
        fit_y = sl * fit_x + ic
        ax.plot(fit_x, fit_y, 'w--', linewidth=1.4, alpha=0.8)

        # Reference lines
        ref_y_05 = 0.5 * fit_x + ic
        ax.plot(fit_x, ref_y_05, color='#ff4444', linestyle=':',
                linewidth=0.9, alpha=0.5, label='H=0.5 (random walk)')

    ax.set_title(f'{name}  (H = {H:.4f}, R² = {r2:.4f})',
                 fontsize=12, fontweight='bold', color=color)
    ax.set_xlabel('log(n)')
    ax.set_ylabel('log(R/S)')
    ax.legend(fontsize=9, framealpha=0.4)
    ax.grid(True)

    # Classification box
    if H > 0.5:
        verdict = 'PERSISTENT'
    elif H < 0.5:
        verdict = 'ANTI-PERSISTENT'
    else:
        verdict = 'RANDOM WALK'
    ax.text(0.03, 0.95, f'H = {H:.4f}\n{verdict}',
            transform=ax.transAxes, fontsize=10, va='top',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#0f0f1a',
                      edgecolor=color, alpha=0.9),
            color='#e0e0ff', fontweight='bold')

plt.tight_layout()
plt.savefig('hurst_rs_loglog.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Saved → hurst_rs_loglog.png')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT — Complexity dashboard (bar comparisons)
# ══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Complexity Metrics Comparison — All Datasets', fontsize=16,
             fontweight='bold', color='#e0e0ff')

names  = list(datasets.keys())
colors = [PALETTE[n] for n in names]

def bar_panel(ax, values, title, ylabel, ref_line=None, ref_label=''):
    bars = ax.barh(names, values, color=colors, edgecolor='#0f0f1a',
                   alpha=0.85, height=0.55)
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + max(values) * 0.02,
                bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=10, color='#e0e0ff')
    if ref_line is not None:
        ax.axvline(ref_line, color='#ff4444', linestyle='--',
                   linewidth=1.0, alpha=0.7, label=ref_label)
        ax.legend(fontsize=8, framealpha=0.4)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel(ylabel)
    ax.grid(True, axis='x')

# (a) ApEn
bar_panel(axes[0,0],
          [datasets[n]['complexity']['apen_m2_r02'] for n in names],
          '(a) Approximate Entropy', 'ApEn (m=2, r=0.2σ)')

# (b) SampEn
sampen_vals = []
for n in names:
    v = datasets[n]['complexity']['sampen_m2_r02']
    sampen_vals.append(v if v != float('inf') else 0)
bar_panel(axes[0,1], sampen_vals, '(b) Sample Entropy', 'SampEn (m=2, r=0.2σ)')

# (c) Hurst (R/S)
bar_panel(axes[0,2],
          [datasets[n]['complexity']['hurst_rs'] for n in names],
          '(c) Hurst Exponent (R/S)', 'H',
          ref_line=0.5, ref_label='H=0.5 (random walk)')

# (d) Hurst (DFA)
bar_panel(axes[1,0],
          [datasets[n]['complexity']['hurst_dfa'] for n in names],
          '(d) Hurst Exponent (DFA)', 'H',
          ref_line=0.5, ref_label='H=0.5 (random walk)')

# (e) Permutation Entropy (order=5)
bar_panel(axes[1,1],
          [datasets[n]['complexity']['perm_entropy_5'] or 0 for n in names],
          '(e) Permutation Entropy (order=5)', 'PE (normalised)',
          ref_line=1.0, ref_label='PE=1 (random)')

# (f) LZ Complexity
bar_panel(axes[1,2],
          [datasets[n]['complexity']['lz_norm'] for n in names],
          '(f) Lempel-Ziv Complexity', 'LZ (normalised)')

plt.tight_layout()
plt.savefig('complexity_dashboard.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Saved → complexity_dashboard.png')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PLOT — Per-dataset detailed panels
# ══════════════════════════════════════════════════════════════════════════════

for name, meta in datasets.items():
    c     = meta['complexity']
    color = PALETTE[name]
    vals  = meta['values']

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    fig.suptitle(f'{name}  —  Complexity Profile',
                 fontsize=15, fontweight='bold', color='#e0e0ff', y=1.02)

    # (a) Time series
    ax = axes[0]
    ax.plot(vals, color=color, alpha=0.9, linewidth=1.2)
    ax.fill_between(range(len(vals)), vals, alpha=0.12, color=color)
    ax.set_title('Time Series', fontsize=12)
    ax.set_xlabel('Sample Index')
    ax.set_ylabel(meta['unit'])
    ax.grid(True)

    # (b) Entropy spider / bar
    ax = axes[1]
    metrics = {
        'ApEn\n(m=2)':   c['apen_m2_r02'],
        'SampEn\n(m=2)': c['sampen_m2_r02'] if c['sampen_m2_r02'] != float('inf') else 0,
        'PE\n(order=3)': c['perm_entropy_3'] or 0,
        'PE\n(order=5)': c['perm_entropy_5'] or 0,
        'LZ\n(norm)':    c['lz_norm'],
    }
    bars = ax.bar(metrics.keys(), metrics.values(), color=color,
                  edgecolor='#0f0f1a', alpha=0.85, width=0.6)
    for bar, val in zip(bars, metrics.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.4f}', ha='center', fontsize=9, color='#e0e0ff')
    ax.set_title('Entropy & Complexity Metrics', fontsize=12)
    ax.set_ylabel('Value')
    ax.grid(True, axis='y')

    # (c) Hurst R/S log-log
    ax = axes[2]
    log_n  = c['hurst_rs_logn']
    log_rs = c['hurst_rs_logrs']
    H      = c['hurst_rs']
    if log_n is not None:
        ax.scatter(log_n, log_rs, color=color, s=50, zorder=5,
                   edgecolors='white', linewidth=0.5)
        from scipy.stats import linregress
        sl, ic, _, _, _ = linregress(log_n, log_rs)
        fit_x = np.linspace(log_n.min(), log_n.max(), 100)
        ax.plot(fit_x, sl * fit_x + ic, 'w--', linewidth=1.4)
        ax.plot(fit_x, 0.5 * fit_x + ic, color='#ff4444', linestyle=':',
                linewidth=0.9, alpha=0.5, label='H=0.5')
    verdict = 'PERSISTENT' if H > 0.5 else 'ANTI-PERSISTENT' if H < 0.5 else 'RANDOM'
    ax.set_title(f'Hurst R/S  (H={H:.4f} → {verdict})', fontsize=12)
    ax.set_xlabel('log(n)')
    ax.set_ylabel('log(R/S)')
    ax.legend(fontsize=9, framealpha=0.4)
    ax.grid(True)

    slug = name.lower().replace(' ', '_').replace('&', 'and')
    plt.tight_layout()
    plt.savefig(f'complexity_{slug}.png', dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.show()
    print(f'Saved → complexity_{slug}.png\n')

In [ ]:
# ── Final cross-dataset summary table ─────────────────────────────────────────
rows = []
for name, meta in datasets.items():
    c = meta['complexity']
    sampen = c['sampen_m2_r02'] if c['sampen_m2_r02'] != float('inf') else None
    rows.append({
        'Dataset':        name,
        'N':              len(meta['values']),
        'ApEn (m=2)':     round(c['apen_m2_r02'], 6),
        'SampEn (m=2)':   round(sampen, 6) if sampen else 'inf',
        'Hurst (R/S)':    round(c['hurst_rs'], 6) if c['hurst_rs'] else 'N/A',
        'Hurst (DFA)':    round(c['hurst_dfa'], 6) if c['hurst_dfa'] else 'N/A',
        'PE (order=5)':   round(c['perm_entropy_5'], 6) if c['perm_entropy_5'] else 'N/A',
        'LZ (norm)':      round(c['lz_norm'], 6),
        'Behaviour':      ('Persistent' if c['hurst_rs'] and c['hurst_rs'] > 0.5 else
                          'Anti-persistent' if c['hurst_rs'] and c['hurst_rs'] < 0.5 else
                          'Random'),
    })

summary_df = pd.DataFrame(rows).set_index('Dataset')
print('\n── Complexity Analysis Summary Table ──')
print(summary_df.to_string())
summary_df

---

## Interpretation Summary

### Entropy (Random vs Structured)
- **Lower ApEn / SampEn** → more regular, more predictable
- **Higher ApEn / SampEn** → more complex, more random-like
- **Permutation Entropy** close to 1 → ordinal patterns are near-random

### Hurst Exponent (Long-term Dependencies)
- **H > 0.5** → **Persistent** — past trends tend to continue (long memory)
- **H = 0.5** → **Random walk** — no predictable long-term pattern
- **H < 0.5** → **Anti-persistent** — mean-reverting, past ups predict future downs

### Dataset Characterisation
- **Dow Jones**: Moderate entropy + high Hurst → structured trending behaviour with some noise
- **S&P 500**: Higher entropy + high Hurst → noisier but still persistent (short data window)
- **Lake Erie**: Low entropy + high Hurst → highly structured with strong long-range seasonal dependence
- **Milk Production**: Lowest entropy + the highest Hurst → most structured and most persistent — clear seasonal determinism

In [ ]:
# ============================================================
# NOTEBOOK TIMER — END
# ============================================================
import time as _timer_module
_NOTEBOOK_END_TIME = _timer_module.time()
_NOTEBOOK_ELAPSED = _NOTEBOOK_END_TIME - _NOTEBOOK_START_TIME
_hours, _rem = divmod(_NOTEBOOK_ELAPSED, 3600)
_minutes, _seconds = divmod(_rem, 60)
print(f"\nTotal notebook execution time: {int(_hours)}h {int(_minutes)}m {_seconds:.2f}s")
print(f"Total seconds: {_NOTEBOOK_ELAPSED:.2f}")
